In [ ]:
import cv2
import os
import time
import uuid
import mediapipe as mp

# Init MediaPipe
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7)
mp_draw = mp.solutions.drawing_utils

images_path = 'dataset/images/train2'
labels = ['yes', 'no','me','you','hello', 'hi', 'good', 'ok', 'sorry', 'i love you', 'thank you', 'welcome']
number_imgs = 100

if not os.path.exists(images_path):
    os.makedirs(images_path)

for label in labels:
    print(f'\nCollecting images for "{label}"')
    label_path = os.path.join(images_path, label)
    os.makedirs(label_path, exist_ok=True)

    print(f'You have 5 seconds to prepare for: "{label}"')
    time.sleep(5)

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Error: Could not open camera.")
        continue

    print(f'Starting capture for "{label}"...\n')
    count = 0

    while count < number_imgs:
        ret, frame = cap.read()
        if not ret:
            print("Failed to grab frame.")
            break

        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(img_rgb)

        if results.multi_hand_landmarks:
            imgname = os.path.join(label_path, f'{label}.{uuid.uuid1()}.jpg')
            cv2.imwrite(imgname, frame)
            print(f'  [{count+1}/{number_imgs}] Saved frame with hand for "{label}"')
            count += 1

            for lm in results.multi_hand_landmarks:
                mp_draw.draw_landmarks(frame, lm, mp_hands.HAND_CONNECTIONS)

        cv2.imshow("Collecting with MediaPipe (press Q to quit)", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        time.sleep(0.4)

    cap.release()
    time.sleep(2)

cv2.destroyAllWindows()
print(f"\n✅ MediaPipe-enhanced image collection complete.")
print("Images saved in:", images_path)